In [1]:
from march.pyt.mc import mc

import torch

In [ ]:
"""
Edge and vertex convention:

                    v4_____________________v5_____________________v10
                    /|                    /|                     /|
                   / |                   / |                    / |
                  /  |                  /  |                   /  |
                 /___|_________________/___|__________________/   |
              v6|    |                 |v7 |                  |v11|
                |    |                 |   |                  |   |
                |    |                 |   |                  |   |
                |    |                 |   |                  |   |
                |    |_________________|___|__________________|___|
                |   / v0               |   / v1               |   / v8
                |  /                   |  /                   |  / 
                | /                    | /                    | /  
                |/_____________________|/_____________________|/
                v2                     v3                     v9
"""

Binary to Offset Table

| Index    | Binray Offset $(z, y, x)$ | Offset $(dx, dy, dz)$ |
| -------- | -------| -------- |
| 0        | 000    | $(0, 0, 0)$
| 1        | 001    | $(1, 0, 0)$
| 2        | 010    | $(0, 1, 0)$
| 3        | 011    | $(1, 1, 0)$
| 4        | 100    | $(0, 0, 1)$
| 5        | 101    | $(1, 0, 1)$
| 6        | 110    | $(0, 1, 1)$
| 7        | 111    | $(1, 1, 1)$

In [5]:
grids = torch.tensor([
    [0, 0, 0], #v0
    [1, 0, 0], #v1
    [0, 1, 0], #v2
    [1, 1, 0], #v3

    [0, 0, 1], #v4
    [1, 0, 1], #v5
    [0, 1, 1], #v6
    [1, 1, 1], #v7

    [2, 0, 0], #v8
    [2, 1, 0], #v9
    [2, 0, 1], #v10
    [2, 1, 1]  #v11
], dtype=torch.float32)

cubes = torch.tensor([
    [0, 1, 2, 3, 4, 5, 6, 7],
    [1, 8, 3, 9, 5, 10, 7, 11]
], dtype=torch.long)

values = torch.tensor(
    [
        1,        #v0
        -0.5,     #v1
        1,        #v2
        -0.5,     #v3
        1,        #v4
        1,        #v5
        1,        #v6
        1,        #v7
        1,        #v8
        1,        #v9
        1,        #v10
        1         #v11
    ], dtype=torch.float32
)

iso = 0.0

verts, faces = mc(grids, cubes, values, iso)

print("# verts:", verts.shape[0])
print("# faces:", faces.shape[0])

print("Verts:")
print(verts)

print("Faces:")
print(faces)

# verts: 6
# faces: 4
Verts:
tensor([[0.6667, 0.0000, 0.0000],
        [1.0000, 1.0000, 0.3333],
        [0.6667, 1.0000, 0.0000],
        [1.0000, 0.0000, 0.3333],
        [1.3333, 1.0000, 0.0000],
        [1.3333, 0.0000, 0.0000]])
Faces:
tensor([[0, 1, 2],
        [3, 1, 0],
        [4, 3, 5],
        [1, 3, 4]])


In [6]:
import plotly.graph_objects as go

# Create figure
fig = go.Figure()

# Create color array: red if value > iso, blue otherwise
colors = ['red' if v > iso else 'blue' for v in values]

# Add grid points as scatter plot
fig.add_trace(go.Scatter3d(
    x=grids[:, 0],
    y=grids[:, 1],
    z=grids[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    name='Grid Points'
))

# Add mesh as surface
fig.add_trace(go.Mesh3d(
    x=verts[:, 0],
    y=verts[:, 1],
    z=verts[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    opacity=0.7,
    color='lightblue',
    name='Mesh'
))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    title='Grid Points and Marching Cubes Mesh',
    width=800,
    height=800
)

fig.show()
